In [ ]:
import sys
sys.path.append("..")
import utils
get_ipython().run_line_magic('matplotlib', 'inline')

import os
import numpy as np
import pandas as pd
import anndata
import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)

# Load data

In [ ]:
metadata = pd.read_excel(
    '../../figures/manuscript_figures/tables/Table1_human_donor_information.xlsx',
    sheet_name='Pancreas', index_col=0)

tech_cols = ["snRNA-Seq", "Immunostaining", "Spatial transcriptomics", "Calcium imaging", "Slice-Seq"]
tech_mask = metadata[tech_cols].astype(str).apply(lambda s: s.str.lower().str.strip().eq("yes"))

metadata["Method"] = tech_mask.apply(
    lambda row: ", ".join([col for col, ok in row.items() if ok]),
    axis=1
)

In [ ]:
fig_folder = '../../figures/spatial/islet_analysis/'
data_folder = '../../data/cell2loc_islet/'
res_folder = '../../results/spatial/islet_analysis/'

os.makedirs(fig_folder, exist_ok=True)
os.makedirs(data_folder, exist_ok=True)
os.makedirs(res_folder, exist_ok=True)

In [16]:
ct_palette = utils.load_ct_palette()
cst_palette = utils.load_cst_palette()

groups = ['ND-Lean', 'ND-Obese', 'Pre-T2D', 'T1D', 'T2D']
group_colors = {group: sns.color_palette('husl', 5)[i] 
                for i, group in enumerate(['ND-Lean', 'ND-Obese', 'Pre-T2D', 'T1D', 'T2D'])}

## load spatial data

In [ ]:
# old version of loading before combining cell2location results to the dataset
spatial = sc.read_h5ad('../../data/raw_data_c2l_merged_endocrine_score_LiSA_islets_hq.h5ad')
spatial = spatial[~ spatial.obs['sample'].isin(['V32'])] # V32 actually is replicate of V31
spatial.raw = spatial

# endocrine bead annotation after cell2location

In [ ]:
spatial_file_low = f"{os.path.join(data_folder, 'spatial/run1_w_U6_slice/epochs2000/LQ_samples/spatial_islet_after_c2l_HQ.h5ad')}"
spatial_file_high = f"{os.path.join(data_folder, 'spatial/run1_w_U6_slice/epochs2000/HQ_samples/spatial_islet_after_c2l_HQ.h5ad')}"

spat_low = sc.read_h5ad(spatial_file_low)
spat_high = sc.read_h5ad(spatial_file_high)
spat_high.obs.loc[spat_high.obs['sample'] == 'U6-slice', 'group'] = 'Pre-T2D'

spat = anndata.concat([spat_low, spat_high], join="outer")

## annotations

In [207]:
from scipy.stats import binom

# ================== 0) input and cleaning ==================
type_cols = ["α","β","γ","δ"]
group_col, sample_col, islet_col = "group", "sample", "islet"

# q05 abundance matrix
abund = pd.DataFrame(
    spat.obsm["q05_cell_abundance_w_sf"].values,
    index=spat.obs_names,
    columns=type_cols
)
type_cols = [c for c in type_cols if c in abund.columns]
assert len(type_cols) == 4, "need exact four columns: α/β/γ/δ"

abund = abund[type_cols].copy()
abund = abund.join(spat.obs[[group_col, sample_col, islet_col]])
print(f"In total {abund.shape[0]} islet beads")

# 5% quantile clipping (zeroing) for each type + calculation of total endocrine volume
abund_filter = abund[type_cols].apply(lambda x: np.quantile(x, 0.05))
abund_clean = abund.copy()
abund_clean[type_cols] = abund_clean[type_cols].mask(abund_clean[type_cols] < abund_filter, 0.0)
abund_clean["endo_total_q05"] = abund_clean[type_cols].sum(axis=1)

# Low quality: The total amount is below the global 5% percentile
enable_low_quality = True
total_q05 = float(np.quantile(abund_clean["endo_total_q05"], 0.05))
low_quality = enable_low_quality & (abund_clean["endo_total_q05"] < total_q05)

# Normalization to get fraction
frac = abund_clean[type_cols].div(abund_clean["endo_total_q05"], axis=0).fillna(0.0)

# Information for ordering
dominant_type = frac.idxmax(axis=1)
dominance = frac.max(axis=1)
second_dominance = frac.apply(lambda r: r.nlargest(2).iloc[-1] if (r > 0).sum() >= 2 else 0.0, axis=1)
third_dominance  = frac.apply(lambda r: r.nlargest(3).iloc[-1] if (r > 0).sum() >= 3 else 0.0, axis=1)
fourth_dominance = frac.min(axis=1)
gap12 = dominance - second_dominance
gap23 = second_dominance - third_dominance
gap34 = third_dominance - fourth_dominance

top1_col = dominant_type
top2_col = frac.apply(lambda r: r.nlargest(2).index[1] if (r > 0).sum() >= 2 else pd.NA, axis=1)
top3_col = frac.apply(lambda r: r.nlargest(3).index[2] if (r > 0).sum() >= 3 else pd.NA, axis=1)

In total 36403 islet beads


In [208]:
# I decide not to apply these corrections because I will smooth or sharpen the data anaway

# null weight: >1 improves the expected value of this type at p0 (harder to be significant)
null_weight = {"α": 1.00, "β": 1.00, "γ": 1.00, "δ": 1.00}
# Statistic weight: <1 reduces the contribution of this type in the S_k statistic
stat_weight = {"α": 1.00, "β": 1.00, "γ": 1.00, "δ": 1.00}
# select 'null' / 'stat' / 'both' / 'none'
weight_mode = "both"

# p0 construction method：'islet' | 'global' | 'equal'
p0_mode = "equal"

# ================== 1) weights and baseline p0 ==================
def apply_null_weight(p0_series: pd.Series) -> pd.Series:
    if weight_mode in ("null", "both"):
        w = pd.Series({k: null_weight.get(k, 1.0) for k in type_cols})
        p = (p0_series * w).clip(1e-12)
        return p / p.sum()
    return p0_series

if p0_mode == "equal":
    base_p0_global = pd.Series([0.25]*4, index=type_cols)
elif p0_mode == "global":
    base_p0_global = (frac[type_cols].mean() / frac[type_cols].mean().sum()).clip(1e-12)
else:  # islet baseline
    base_p0_global = (frac[type_cols].mean() / frac[type_cols].mean().sum()).clip(1e-12)

if p0_mode == "islet":
    p0_by_islet = frac.join(abund_clean[[islet_col]]).groupby(islet_col)[type_cols].mean()
    p0_by_islet = p0_by_islet.div(p0_by_islet.sum(axis=1), axis=0).fillna(base_p0_global)
    p0_by_islet = p0_by_islet.apply(apply_null_weight, axis=1)
else:
    p0_by_islet = None
    base_p0_global = apply_null_weight(base_p0_global)

def get_p0(idx):
    if p0_mode == "islet":
        isl = abund_clean.loc[idx, islet_col]
        p0 = p0_by_islet.loc[isl] if isl in p0_by_islet.index else base_p0_global
    else:
        p0 = base_p0_global
    return p0.astype(float).clip(1e-12)

def apply_stat_weight_vec(vec: np.ndarray) -> np.ndarray:
    if weight_mode in ("stat", "both"):
        w = np.array([stat_weight.get(k, 1.0) for k in type_cols], dtype=float)
        return vec * w
    return vec

# ================== 2) significance test ==================
N_eff = 100           # Pseudo sample size when there is no true count
n_perm = 1000         # Number of replacements (if you need faster, you can start with 300-500)
rng = np.random.default_rng(123)

def holm_bonferroni(pvals: pd.Series) -> pd.Series:
    # Returns the Holm-Bonferroni adjusted p-values in the same index order as pvals
    order = pvals.sort_values()
    m = len(order)
    adj_sorted = np.array([(m - i) * p for i, p in enumerate(order.values)], dtype=float)
    adj_sorted = np.maximum.accumulate(adj_sorted)  # Preserving order
    adj = pd.Series(adj_sorted, index=order.index).reindex(pvals.index)
    return adj.clip(upper=1.0)

def test_top1_holm(row_f: pd.Series, idx) -> float:
    p0 = get_p0(idx)
    counts = (row_f * N_eff).round().astype(int)
    pvals = pd.Series(index=type_cols, dtype=float)
    for t in type_cols:
        x = int(counts[t])
        p0_i = float(p0[t])
        pvals[t] = 1.0 - binom.cdf(x-1, N_eff, p0_i)  # Unilateral upper tail
    adj = holm_bonferroni(pvals)
    top1 = row_f.idxmax()
    return float(adj[top1])

def perm_pvalue_Sk(row_f: pd.Series, idx, k=2) -> float:
    p0 = get_p0(idx).values
    obs_counts = (row_f.values * N_eff).round().astype(int)
    obs_w = apply_stat_weight_vec(obs_counts.astype(float))
    S_obs = np.sort(obs_w)[::-1][:k].sum()
    sims = rng.multinomial(N_eff, p0, size=n_perm)
    sims_w = apply_stat_weight_vec(sims.astype(float))
    S_sim = np.sort(sims_w, axis=1)[:, ::-1][:, :k].sum(axis=1)
    pval = (1 + (S_sim >= S_obs).sum()) / (n_perm + 1)
    return float(pval)

# Compute p-values row by row
top1_adj_p = pd.Series(index=frac.index, dtype=float)
S2_pval = pd.Series(index=frac.index, dtype=float)
S3_pval = pd.Series(index=frac.index, dtype=float)

for idx, row in frac[type_cols].iterrows():
    top1_adj_p[idx] = test_top1_holm(row, idx)
    S2_pval[idx] = perm_pvalue_Sk(row, idx, k=2)
    S3_pval[idx] = perm_pvalue_Sk(row, idx, k=3)

In [213]:
# ================== 3) Based on salience + Ambiguous only when "very close" ==================
alpha_top1 = 0.05
alpha_S2   = 0.05
alpha_S3   = 0.05

# "Very close" threshold (two ways to write, choose one)
close_span = 0.12   # Solution A: max-min < 0.12 is considered close (recommended, simple and robust)
# Or use small adjacent differences:
close_gap  = 0.08   # Option B: adjacent differences are all < 0.08

def is_very_close_row(a, b, c, d):
    # SolutionA（default）：based on max-min 
    vals = np.array([a, b, c, d], dtype=float)
    return (vals.max() - vals.min()) < close_span

    # If you want to use solution B instead, comment out the return statement above and change it to:
    # gaps_ok = (abs(a-b) < close_gap) and (abs(b-c) < close_gap) and (abs(c-d) < close_gap)
    # return gaps_ok

# Ordered naming of mixed tags
mix_type_order = ["α","β","γ","δ"]
mix_type_rank = {k: i for i, k in enumerate(mix_type_order)}
def ordered_pair(a, b):
    if pd.isna(a) or pd.isna(b): return pd.NA
    key = lambda x: mix_type_rank.get(x, 1e9)
    first, second = sorted([a, b], key=key)
    return f"{first}_{second}"
def ordered_triplet(a, b, c):
    if pd.isna(a) or pd.isna(b) or pd.isna(c): return pd.NA
    key = lambda x: mix_type_rank.get(x, 1e9)
    first, second, third = sorted([a, b, c], key=key)
    return f"{first}_{second}_{third}"

label = pd.Series(index=frac.index, dtype="object")

# 1) Label "Low quality" first 
label[low_quality] = "Low quality"
eligible = ~low_quality

# 2) Simplicity first：single --> 2-mix --> 3-mix
# 2.1 single
remaining = eligible & (label.isna())
is_single = remaining & (top1_adj_p <= alpha_top1)
label[is_single] = dominant_type[is_single]

# 2.2 2-mix
remaining = remaining & (label.isna())
in_2mix = remaining & (S2_pval <= alpha_S2)
label[in_2mix] = [
    ordered_pair(a, b)
    for a, b in zip(top1_col[in_2mix], top2_col[in_2mix])
]

# 2.1 3-mix
remaining = remaining & (label.isna())
in_3mix = remaining & (S3_pval <= alpha_S3)
label[in_3mix] = [
    ordered_triplet(a, b, c)
    for a, b, c in zip(top1_col[in_3mix], top2_col[in_3mix], top3_col[in_3mix])
]

# 3) For those who have not yet marked it: mark it as Ambiguous only when "the four types are very close", 
remaining = eligible & (label.isna())
if remaining.any():
    # Determine whether it is "very close" line by line
    close_mask = []
    for idx in remaining[remaining].index:
        a = dominance.loc[idx]
        b = second_dominance.loc[idx]
        c = third_dominance.loc[idx]
        d = fourth_dominance.loc[idx]
        close_mask.append(is_very_close_row(a, b, c, d))
    close_mask = pd.Series(close_mask, index=remaining[remaining].index)

    # very close -> Ambiguous
    amb_idx = close_mask[close_mask].index
    label.loc[amb_idx] = "α_β_γ_δ"
    non_close_idx = close_mask[~close_mask].index

In [214]:
remaining_nc = pd.Index(non_close_idx)

# ----- non-close resolution: prefer 3-mix vs 2-mix by p-values + a clear post-k gap -----
if len(remaining_nc):
    
    # 2-mix if S2 wins AND drop to #3 is clear (gap23 >= min_gap)
    cond_s2_loose = (S3_pval.loc[remaining_nc] > S2_pval.loc[remaining_nc])

    idx_2mix_nc = remaining_nc[cond_s2_loose.fillna(False)]
    if len(idx_2mix_nc):
        label.loc[idx_2mix_nc] = [
            ordered_pair(a, b)
            for a, b in zip(top1_col.loc[idx_2mix_nc],
                            top2_col.loc[idx_2mix_nc])
        ]
        
    # remaining is 3-mix assignment
    idx_3mix_nc = remaining_nc.difference(idx_2mix_nc)

    if len(idx_3mix_nc):
        label.loc[idx_3mix_nc] = [
            ordered_triplet(a, b, c)
            for a, b, c in zip(top1_col.loc[idx_3mix_nc],
                               top2_col.loc[idx_3mix_nc],
                               top3_col.loc[idx_3mix_nc])
        ]

In [ ]:
# ================== 4) summary ==================
annot = pd.DataFrame({
    "group": abund_clean[group_col],
    "sample": abund_clean[sample_col],
    "islet":  abund_clean[islet_col],
    "endo_total_q05": abund_clean["endo_total_q05"],
    "label": label,
    "top1_adj_p": top1_adj_p,
    "S2_pval": S2_pval,
    "S3_pval": S3_pval,
    "dominant": dominant_type,
}, index=frac.index)

for ct in type_cols:
    annot[f"{ct}_abs"]  = abund_clean[ct]
    annot[f"{ct}_frac"] = frac[ct]

print("\n--- Low-quality threshold (q05 on total) ---")
print(f"{'enabled' if enable_low_quality else 'disabled'}; cutoff={total_q05:.6f}")
print("\n--- Weight modes ---")
print(f"p0_mode={p0_mode}, weight_mode={weight_mode}")
print("null_weight=", null_weight)
print("stat_weight=", stat_weight)
print("\n--- Label counts ---")
print(annot["label"].value_counts(dropna=False).sort_index().to_string())

# add the cell2location to spatial_islet
for col in ['α_frac', 'β_frac', 'γ_frac', 'δ_frac', 'endo_total_q05', 'label',]:
    spat.obs.loc[annot.index, col] = annot[col]
spat.write(os.path.join(data_folder, 'spatial_islet_after_cell2loc_annotation.h5ad'))


--- Low-quality threshold (q05 on total) ---
enabled; cutoff=0.174465

--- Weight modes ---
p0_mode=equal, weight_mode=both
null_weight= {'α': 1.0, 'β': 1.0, 'γ': 1.0, 'δ': 1.0}
stat_weight= {'α': 1.0, 'β': 1.0, 'γ': 1.0, 'δ': 1.0}

--- Label counts ---
label
Low quality     1821
α              15187
α_β              972
α_β_γ            403
α_β_γ_δ         1154
α_β_δ            617
α_γ              435
α_γ_δ            290
α_δ              594
β              10706
β_γ              238
β_γ_δ            141
β_δ              428
γ               1476
γ_δ              153
δ               1788


### relabel mixed spots

In [ ]:
from sklearn.neighbors import NearestNeighbors

# ---------------- Shared neighbor & voting utilities ----------------
def _neighbors_within_islet(adata, i_global, group_key, coord_key, k, r_auto, min_need):
    '''
    Find the closest neighbors of a given spot, restricted to the same islet.
    
    Parameters
    ----------
    adata : AnnData
        The spatial dataset containing `.obs` metadata and `.obsm[coord_key]` coordinates.
    i_global : int
        The global index (row number in `adata.obs_names`) of the focal spot.
    group_key : str
        Column in `.obs` defining islet membership (e.g. "islet").
    coord_key : str
        Key in `.obsm` containing spatial coordinates (e.g. "spatial").
    k : int or None
        Maximum number of nearest neighbors to return (cap). If None, only radius is used.
    r_auto : float or None
        Precomputed radius for this islet. If None, radius is estimated automatically
        as ≈ 2.5 × median 2nd-nearest-neighbor spacing within the islet.
    min_need : int
        Minimum number of neighbors desired. Used to ensure we query enough
        candidates even if k is small.

    Returns
    -------
    gi : ndarray of int
        Global indices of all spots in the same islet as the focal spot.
    ind : ndarray of int
        Local indices (into `Xi`) of the chosen neighbors of the focal spot,
        after applying radius and k cap.
    Xi : ndarray of shape (n_islet, 2)
        Coordinates of all spots in the islet.
    i_local : int
        Local index of the focal spot within `Xi`.
    r_use : float or None
        Radius actually used for filtering neighbors of this spot.
    g : str
        The islet ID of the focal spot.
    '''
    obs = adata.obs
    coords = np.asarray(adata.obsm[coord_key], dtype=float) # coordiates of all spots
    g = obs.iloc[i_global][group_key] # islet ID of the spot
    gi = np.where(obs[group_key].astype(str).values == str(g))[0] # all global indices of spots in that islet
    if gi.size < 2: # avoid islet with only 1 spot
        return gi, np.array([], dtype=int), None, None, None, g

    Xi = coords[gi] # coordinates of spots in that islet
    loc = np.where(gi == i_global)[0] # the global index of a chosen spot
    if loc.size == 0: # the chosen spot’s index wasn’t found inside gi
        return gi, np.array([], dtype=int), None, None, None, g
    i_local = loc[0] # the position of the spot within its islet’s local coordinate array

    # auto radius per-islet if None: ≈ 2.5 × median 2nd-NN spacing
    if r_auto is None: # if no spefified radius
        # For each spot in Xi, find its 2 nearest neighbors (n_neighbors=3 --> itself + its 2 closest neighbors).
        nn_small = NearestNeighbors(n_neighbors=min(3, len(gi))).fit(Xi)
        d, _ = nn_small.kneighbors(Xi) # distances to those neighbors
        spacing = float(np.median(d[:, 1])) if Xi.shape[0] >= 3 else 0.0 # take the 2nd smallest distance for each spot
        r_use = 2.5 * spacing if spacing > 0 else None
    else:
        r_use = float(r_auto)

    # build larger NN to be safe; we’ll filter by radius then cap to k
    want = (k or min_need) + 1
    nn = NearestNeighbors(n_neighbors=min(max(want, 15), len(gi))).fit(Xi)
    d, ind = nn.kneighbors(Xi[i_local:i_local+1], n_neighbors=min(want, len(gi)))
    d = d.ravel()[1:]; ind = ind.ravel()[1:]  # drop self

    if r_use is not None:
        keep = d <= r_use
        d = d[keep]; ind = ind[keep]

    if (k is not None) and (ind.size > k):
        d = d[:k]; ind = ind[:k]

    return gi, ind, Xi, i_local, r_use, g

def _normalize_indices(candidates_idx, n_total):
    if candidates_idx is None:
        return np.arange(n_total, dtype=int)
    return np.asarray(candidates_idx, dtype=int).ravel()

def simple_relabel_pass(
    adata, labels,
    *,                         # keyword-only
    mode,                      # "smooth" or "sharpen"
    sep="_",
    group_key="islet",
    coord_key="spatial",
    k=10,
    radius=None,
    min_neighbors_lo=2,        # if neighbors <= this → expand radius once
    max_expansions=6,          # safety cap
    candidates_idx=None,       # optional: iterable of indices to process
    ignored_labels=None,       # iterable of label strings to skip relabeling
    frac=None,                 # optional DataFrame: rows=spot, cols=parts (e.g. α,β,γ,δ)
):
    """
    Simple relabeling:
      - smooth: choose the part with the most neighbor support (majority)
      - sharpen: choose the part with the least neighbor support (minority)
    If neighbors <= min_neighbors_lo, expand radius maxium max_expansions times by one spacing and retry.

    Returns
    -------
    labels_out : pd.Series
    changes_df : pd.DataFrame
    no_changes_df : pd.DataFrame
    """

    if mode not in {"smooth", "sharpen"}:
        raise ValueError("mode must be 'smooth' or 'sharpen'")

    ignored = set(map(str, ignored_labels)) if ignored_labels else set()
    has_frac = frac is not None

    labels_out = labels.copy()
    idx_all = adata.obs_names
    changes, no_changes = [], []

    iter_indices = _normalize_indices(candidates_idx, len(idx_all))

    for i_global in iter_indices:
        spot_id = idx_all[i_global]
        lab = labels_out.iat[i_global]

        # skip if ignored
        if lab in ignored:
            no_changes.append({
                "spot": spot_id, "old": lab, "new": lab,
                "reason": "ignored_label", "frac": None,
                "nN": None, "anchors": None, "cluster_size": None
            })
            continue

        parts = [p for p in lab.split(sep) if p]
        if len(parts) < 2:
            continue  # only relabel mixed spots

        # neighborhood (iteratively expand)
        gi, loc, Xi, i_local, r_use, _ = _neighbors_within_islet(
            adata, i_global, group_key, coord_key, k, radius, min_neighbors_lo
        )
        nN = int(loc.size) if (loc is not None and hasattr(loc, "size")) else 0

        expansions = 0
        base_r = (radius if radius else (r_use if r_use else 0.0))
        spacing = (r_use / 2.5) if (r_use and r_use > 0) else None

        while (Xi is None or nN < min_neighbors_lo) and expansions < max_expansions:
            if spacing is None:
                break
            expansions += 1
            new_r = base_r + expansions * spacing
            gi, loc, Xi, i_local, r_use2, _ = _neighbors_within_islet(
                adata, i_global, group_key, coord_key, k, new_r, min_neighbors_lo
            )
            nN = int(loc.size) if (loc is not None and hasattr(loc, "size")) else 0

        if Xi is None or nN < min_neighbors_lo:
            no_changes.append({
                "spot": spot_id, "old": lab, "new": lab,
                "reason": f"simple_too_few_neighbors_after_iter_expand(nN={nN},exp={expansions})",
                "frac": None, "nN": nN, "anchors": None, "cluster_size": None
            })
            continue

        # neighbor votes on the focal parts
        neigh_idx  = gi[loc]
        neigh_labs = labels_out.iloc[neigh_idx].tolist()
        votes = {p: 0.0 for p in parts}
        for nl in neigh_labs:
            n_parts = [p for p in nl.split(sep) if p]
            w = 1.0 / max(len(n_parts), 1)
            for p in n_parts:
                if p in votes:
                    votes[p] += w

        total_votes = sum(votes.values())

        # focal fractions (for tie / no-vote fallback)
        foc_frac = None
        if has_frac and (spot_id in frac.index):
            row = frac.loc[spot_id]
            foc_frac = {p: float(row.get(p, 0.0)) for p in parts}
        sum_foc = (sum(foc_frac.values()) if foc_frac is not None else 0.0)

        out_frac = None  # <- ensure initialized

        # no neighbor votes → fallback to focal fractions
        if total_votes <= 0:
            if not foc_frac or sum_foc == 0.0:
                no_changes.append({
                    "spot": spot_id, "old": lab, "new": lab,
                    "reason": "simple_no_votes_no_frac_fallback",
                    "frac": None, "nN": nN, "anchors": None, "cluster_size": None
                })
                continue
            if mode == "smooth":
                target = max(parts, key=lambda p: foc_frac[p])
                out_frac = foc_frac[target] / max(sum_foc, 1e-12)
                reason = "simple_no_votes_fallback_focal_max"
            else:
                target = max(parts, key=lambda p: foc_frac[p])  # tie-break rule = higher frac
                # but since sharpen wants the minority conceptually, you already decided
                # to always use "higher frac" for ties/fallback, so keep consistent:
                out_frac = foc_frac[target] / max(sum_foc, 1e-12)
                reason = "simple_no_votes_fallback_focal_min"  # name kept for provenance
            if lab != target:
                changes.append({
                    "spot": spot_id, "old": lab, "new": target,
                    "reason": reason, "frac": float(out_frac),
                    "nN": nN, "anchors": None, "cluster_size": None
                })
                labels_out.iat[i_global] = target
            else:
                no_changes.append({
                    "spot": spot_id, "old": lab, "new": lab,
                    "reason": reason.replace("fallback", "fallback_already_consistent"),
                    "frac": float(out_frac), "nN": nN, "anchors": None, "cluster_size": None
                })
            continue

        # we have votes
        ranked = sorted(votes.items(), key=lambda kv: kv[1], reverse=True)
        top_p, top_v = ranked[0]
        bot_p, bot_v = ranked[-1]
        tie_top = sum(v == top_v for _, v in votes.items()) > 1
        tie_bot = sum(v == bot_v for _, v in votes.items()) > 1

        if mode == "smooth":
            if tie_top and foc_frac and sum_foc > 0:
                target = max([p for p, v in votes.items() if v == top_v], key=lambda p: foc_frac[p])
            else:
                target = None if tie_top else top_p
            out_frac = (votes[target] / total_votes) if target else (top_v / total_votes)
            reason_keep = "simple_tie_majority"
            reason_change = "simple_majority"
        else:  # sharpen (minority), ties also resolved by higher frac (your rule)
            if tie_bot and foc_frac and sum_foc > 0:
                target = max([p for p, v in votes.items() if v == bot_v], key=lambda p: foc_frac[p])
            else:
                target = None if tie_bot else bot_p
            out_frac = (votes[target] / total_votes) if target else (bot_v / total_votes)
            reason_keep = "simple_tie_minority"
            reason_change = "simple_minority"

        if target is None:
            no_changes.append({
                "spot": spot_id, "old": lab, "new": lab,
                "reason": reason_keep, "frac": float(out_frac) if out_frac is not None else None,
                "nN": nN, "anchors": None, "cluster_size": None
            })
            continue

        # ensure out_frac is set for change logging
        if out_frac is None:
            out_frac = (votes.get(target, 0.0) / total_votes) if total_votes > 0 else None

        if lab != target:
            labels_out.iat[i_global] = target
            changes.append({
                "spot": spot_id, "old": lab, "new": target,
                "reason": reason_change, "frac": float(out_frac) if out_frac is not None else None,
                "nN": nN, "anchors": None, "cluster_size": None
            })
        else:
            no_changes.append({
                "spot": spot_id, "old": lab, "new": lab,
                "reason": "simple_already_consistent",
                "frac": float(out_frac) if out_frac is not None else None,
                "nN": nN, "anchors": None, "cluster_size": None
            })

    changes_df = pd.DataFrame(changes, columns=["spot","old","new","reason","frac","nN","anchors","cluster_size"])
    no_changes_df = pd.DataFrame(no_changes, columns=["spot","old","new","reason","frac","nN","anchors","cluster_size"])
    return labels_out, changes_df, no_changes_df

In [ ]:
new_labels_sharpen, changes_sharpen, no_changes_sharpen = simple_relabel_pass(
    spat,
    spat.obs["label"].astype(str).copy(),
    mode="sharpen",
    ignored_labels={"α_β"},  # skip relabeling these
    frac=frac
)
spat.obs["dominant_sharpen"] = new_labels_sharpen

spat.write(os.path.join(data_folder, 'spatial_islet_after_cell2loc_annotation.h5ad'))

# islet feature generation

In [8]:
from scipy.spatial import ConvexHull, KDTree
from sklearn.decomposition import PCA

def pca_axes_eccentricity(X):
    """
    Return major_axis, minor_axis, orientation (radians), eccentricity from 2D coords.
    """
    if X.shape[0] < 3:
        return np.nan, np.nan, np.nan, np.nan
    p = PCA(n_components=2).fit(X)
    # singular values ~ sqrt(eigenvalues) scaled by sqrt(n-1)
    s = p.singular_values_
    major, minor = (s[0], s[1]) if s[0] >= s[1] else (s[1], s[0])
    # orientation of the first PC (angle of major axis)
    angle = np.arctan2(p.components_[0,1], p.components_[0,0])
    if major == 0: 
        ecc = np.nan
    else:
        ecc = np.sqrt(1 - (minor/major)**2)  # ellipse eccentricity
    return major, minor, angle, ecc

def hull_area_perimeter(X):
    """
    Convex hull area and perimeter; returns (area, perimeter).
    """
    if X.shape[0] < 3:
        return np.nan, np.nan
    hull = ConvexHull(X) # a geometric tool that computes the convex hull of a set of points
    A = hull.area if X.shape[1]==2 else np.nan 
    # SciPy quirk: in 2D, ConvexHull.area = perimeter, ConvexHull.volume = area
    area = hull.volume
    # compute perimeter from hull vertices
    verts = hull.vertices # indices of the corner points that make up the convex hull
    P = 0.0
    for i in range(len(verts)):
        a = X[verts[i]] # current vertex (x1, y1)
        b = X[verts[(i+1) % len(verts)]] # next vertex (x2, y2)
        P += np.linalg.norm(a - b) # Euclidean distance between the two points, np.sqrt((x1-x2)**2 + (y1 -y2)**2)
    return area, P

def circularity(area, perimeter):
    """
    compactness; 1--> round
    """
    if np.isnan(area) or np.isnan(perimeter) or perimeter == 0:
        return np.nan
    return 4*np.pi*area/(perimeter**2)
7 
def shannon_diversity(counts):
    """
    Shannon diversity is a way of quantifying mixture / heterogeneity
    
    paramter: raw counts of each category (cell type), e.g., [10, 30, 5, 5]
    
    returen:
        H=−∑pi​ln(pi) -- pi is the pct of ct i

        Low H (close to 0): dominated by one cell type (e.g., 95% β).
        High H (up to ln N, where N = number of types): more even mix.
            For 4 endocrine subtypes, max 𝐻=ln4≈1.386 (when all are 25%).
    """
    counts = np.asarray(counts, dtype=float)
    n = counts.sum()
    if n == 0:
        return 0.0
    p = counts[counts>0]/n
    return -(p*np.log(p)).sum()

def mean_gene(adata, bead_idx, gene):
    if gene not in adata.var_names:
        return np.nan
    vals = adata[bead_idx, gene].X
    vals = vals.toarray() if hasattr(vals, "toarray") else (vals.A if hasattr(vals, "A") else vals)
    return np.asarray(vals).ravel().mean()

def distance_to_nearest_class(X_in, X_out):
    if X_in.shape[0] == 0 or X_out.shape[0] == 0:
        return np.nan
    tree = KDTree(X_out)
    d, _ = tree.query(X_in, k=1)
    return float(np.mean(d))

# ---------- NEW: weights + weighted core/mantle ----------
def build_type_weights(obs_names, labels, frac, type_cols=("α","β","γ","δ"), 
                       sep="_", equal_split_if_missing=True):
    """
    Returns a DataFrame W (n_spots x |type_cols|) with per-spot weights that sum to 1.0 over the parts in the label:
      - Single 'α' → row has α=1.0, others=0
      - Mixed 'α_β' → row has α=fα/(fα+fβ), β=fβ/(fα+fβ), others=0
      - If frac missing for a part and equal_split_if_missing=True → equal split among parts
    """
    W = pd.DataFrame(0.0, index=obs_names, columns=list(type_cols))
    labels = pd.Series(labels, index=obs_names, dtype=str)

    for spot, lab in labels.items():
        parts = [p for p in lab.split(sep) if p]
        parts = [p for p in parts if p in type_cols]
        if len(parts) == 0:
            continue
        if len(parts) == 1:
            W.at[spot, parts[0]] = 1.0
            continue

        if (frac is not None) and (spot in frac.index):
            vals = pd.Series({p: float(frac.at[spot, p]) if p in frac.columns else 0.0 for p in parts})
            s = vals.sum()
            if s > 0:
                vals = vals / s
                for p, w in vals.items():
                    W.at[spot, p] = w
                continue

        # fallback: equal split
        if equal_split_if_missing:
            w = 1.0 / len(parts)
            for p in parts:
                W.at[spot, p] = w

    return W

def core_mantle_fraction_weighted(X, weights_rowvec, inner_quantile=0.5):
    """
    Weighted fraction in core vs mantle for a single target type's weights.
      - weights_rowvec: shape (n_spots,), non-negative, not necessarily summing to 1 across spots.
      - Returns (core_frac, mantle_frac, diff) where each fraction is mean(weight) per bead in region.
        (i.e., sum(weights_region) / n_beads_region), mirroring unweighted indicator mean.
    """
    n = X.shape[0]
    if n == 0:
        return np.nan, np.nan, np.nan
    c = X.mean(axis=0)
    r = np.linalg.norm(X - c, axis=1)
    r_thr = np.quantile(r, inner_quantile)
    core_mask = (r <= r_thr)
    mantle_mask = ~core_mask
    n_core, n_mantle = int(core_mask.sum()), int(mantle_mask.sum())
    if n_core == 0 or n_mantle == 0:
        return np.nan, np.nan, np.nan
    core_frac = float(weights_rowvec[core_mask].sum()) / n_core
    mantle_frac = float(weights_rowvec[mantle_mask].sum()) / n_mantle
    return core_frac, mantle_frac, core_frac - mantle_frac

# ---------- UPDATED: compute_islet_features (weighted) ----------
def compute_islet_features(
    adata,
    spatial_samples,
    *,
    ctype_col="max_pred_celltype",     # your subtype annotation column (strings like 'α','α_β',…)
    coords=("x", "y"),
    endocrine_labels=('α', 'β', 'γ', 'δ'),
    neighbor_labels=("Ductal", "Endothelial", "Acinar", "Immune"),
    frac=None,                         # NEW: DataFrame (rows=spot, cols=endocrine types), used for mixed weights
    sep="_"
):
    rows = []
    endocrine_labels = tuple(endocrine_labels)  # ensure tuple

    for sample in spatial_samples:
        spatial_tmp = adata[adata.obs['sample'] == sample].copy()
        islet_ids = spatial_tmp.obs['islet'].dropna().unique()

        for iso in islet_ids:
            sel = (spatial_tmp.obs['islet'] == iso)
            idx = np.where(sel)[0]
            if idx.size == 0:
                continue

            spot_ids = spatial_tmp.obs_names[idx]
            ct_all   = spatial_tmp.obs[ctype_col].astype(str).values[idx] # labels from annotation
            X        = spatial_tmp.obs[list(coords)].values[idx].astype(float)

            # ---- geometry
            N = idx.size
            area, perim = hull_area_perimeter(X)
            circ = circularity(area, perim)
            major, minor, theta, ecc = pca_axes_eccentricity(X)

            centroid = X.mean(axis=0)
            radii = np.linalg.norm(X - centroid, axis=1)
            rad_mean = float(np.mean(radii))
            rad_median = float(np.median(radii))
            rad_max = float(np.max(radii))
            rad_cv = float(np.std(radii)/np.mean(radii)) if np.mean(radii) > 0 else np.nan
            density = N/area if area and not np.isnan(area) and area > 0 else np.nan

            # ---- composition via WEIGHTS
            # Build per-spot weights for endocrine labels using labels + frac
            # Note: pass frac restricted to current adata (global accepted too)
            frac_df = frac if frac is not None else None
            W = build_type_weights(spot_ids, pd.Series(ct_all, index=spot_ids),
                                   frac=frac_df, type_cols=endocrine_labels, sep=sep)

            # Sample-level (islet-level here) weighted counts & fractions
            weighted_counts = W.sum(axis=0).reindex(endocrine_labels).fillna(0.0).values
            total_weight = weighted_counts.sum()  # should equal N if labels are all endocrine; else ≤ N
            weighted_fracs = (weighted_counts / N) if N > 0 else np.full(len(endocrine_labels), np.nan)

            a_f, b_f, g_f, d_f = (float(weighted_fracs[i]) for i in range(len(endocrine_labels)))

            # Endocrine fraction: average of total endocrine weight per bead
            # (beads that are non-endocrine or ambiguous get <1 contribution)
            endocrine_weight_per_bead = W.sum(axis=1).reindex(spot_ids).fillna(0.0).values
            frac_endocrine = float(endocrine_weight_per_bead.mean()) if N > 0 else np.nan

            # Shannon diversity on weighted counts
            H = shannon_diversity(weighted_counts)
            dominance = float(weighted_counts.max()/weighted_counts.sum()) if weighted_counts.sum() > 0 else np.nan

            # ---- core–mantle (weighted)
            # Use type-specific weight vectors over beads
            beta_w = W['β'].reindex(spot_ids).fillna(0.0).values
            alpha_w = W['α'].reindex(spot_ids).fillna(0.0).values

            beta_core, beta_mantle, beta_core_minus_mantle = core_mantle_fraction_weighted(X, beta_w, inner_quantile=0.5)
            alpha_core, alpha_mantle, alpha_core_minus_mantle = core_mantle_fraction_weighted(X, alpha_w, inner_quantile=0.5)

#             # ---- expression summaries
#             INS_mean = mean_gene(spatial_tmp, idx, "INS")
#             GCG_mean = mean_gene(spatial_tmp, idx, "GCG")
#             SST_mean = mean_gene(spatial_tmp, idx, "SST")
#             PPY_mean = mean_gene(spatial_tmp, idx, "PPY")
#             ins_gcg_ratio = INS_mean/GCG_mean if (isinstance(GCG_mean, (int,float)) and GCG_mean not in (0, np.nan) and not np.isnan(GCG_mean)) else np.nan

#             # ---- neighbor proximity (same as before)
#             neighbor_stats = {}
#             for nb in neighbor_labels:
#                 nb_mask = (spatial_tmp.obs[ctype_col].astype(str) == nb) & (~sel)
#                 X_nb = spatial_tmp.obs[list(coords)].values[nb_mask].astype(float)
#                 neighbor_stats[f"nn_dist_to_{nb}"] = distance_to_nearest_class(X, X_nb)

            rows.append(dict(
                sample=sample,
                islet_id=iso,
                n_beads=N,
                area=area,
                perimeter=perim,
                circularity=circ,
                major_axis=major,
                minor_axis=minor,
                orientation_rad=theta,
                eccentricity=ecc,
                radius_mean=rad_mean,
                radius_median=rad_median,
                radius_max=rad_max,
                radius_cv=rad_cv,
                bead_density=density,

                # weighted composition
                frac_endocrine=frac_endocrine,
                alpha_frac=a_f, beta_frac=b_f, gamma_frac=g_f, delta_frac=d_f,
                endocrine_shannon=H,
                endocrine_dominance=dominance,

                # weighted core–mantle
                beta_core_frac=beta_core,
                beta_mantle_frac=beta_mantle,
                beta_core_minus_mantle=beta_core_minus_mantle,
                alpha_core_frac=alpha_core,
                alpha_mantle_frac=alpha_mantle,
                alpha_core_minus_mantle=alpha_core_minus_mantle,

#                 # expression & neighbors
#                 INS_mean=INS_mean, GCG_mean=GCG_mean, SST_mean=SST_mean, PPY_mean=PPY_mean,
#                 INS_over_GCG=ins_gcg_ratio,
#                 **neighbor_stats
            ))

    return pd.DataFrame(rows)

In [ ]:
spat = sc.read_h5ad('../data/cell2loc_islet/spatial_islet_after_cell2loc_annotation.h5ad')

spat_norm = spat.copy()

sc.pp.filter_genes(spat_norm, min_cells=10)
sc.pp.normalize_total(spat_norm, target_sum=1e4, inplace=True)  # CP10K scale
sc.pp.log1p(spat_norm, base=2) # log2 transform 

filtered out 648 genes that are detected in less than 10 cells
normalizing counts per cell
    finished (0:00:01)


In [ ]:
# ---- run it ----
spatial_samples = spat_norm.obs['sample'].unique().tolist()

features = compute_islet_features(
    adata=spat_norm,
    spatial_samples=spatial_samples,
    ctype_col="label",   # change if your column name differs
    coords=("x","y"),
    endocrine_labels=('α', 'β', 'γ', 'δ'),
    neighbor_labels=("Acinar", "Ductal","Endothelial","Stellate", "Immune", "Schwann")
)

# features = features.drop([
#     'nn_dist_to_Acinar', 'nn_dist_to_Ductal', 'nn_dist_to_Endothelial',
#     'nn_dist_to_Stellate', 'nn_dist_to_Immune', 'nn_dist_to_Schwann'], axis=1)

features.loc[features['area'] >= 12000, 'area_group'] = 'Large'
features.loc[(features['area'] >= 2600) & (features['area'] < 12000), 'area_group'] = 'Medium'
features.loc[features['area'] < 2600, 'area_group'] = 'Small'

# Convert area to approximate cell number
features["n_cells"] = features["area"] / 170.0
# Apply log2 transform and round down
features["eo_bin"] = np.floor(np.log2(features["n_cells"].fillna(0).clip(lower=1))).astype(int)
features["eo_bin"] = np.maximum(features["eo_bin"], 0)

for col in ['Sex', 'BMI', 'Diabetes Duration', 
            'HbA1c (%)', 'HbA1c', 'Age', 'group']:
    features[col] = metadata.loc[features['sample'].str.split('-').str[0]][col].tolist()
    
features.to_csv(os.path.join(res_folder, 'islet_features.tsv'), sep='\t', index=False)

# combine the annotation of islets into original dataset

In [ ]:
spat = sc.read_h5ad('../data/cell2loc_islet/spatial_islet_after_cell2loc_annotation.h5ad')

spatial.obs.loc[spat.obs_names, 'max_celltype'] =  'Endocrine'
spatial.obs.loc[spat.obs_names, 'max_cellsubtype'] =  spat.obs[['α_frac', 'β_frac', 'γ_frac', 'δ_frac']].idxmax(axis=1).str[0]

spatial.obs.loc[spat.obs_names, 'islet_label'] =  spat.obs['label']
spatial.obs.loc[spat.obs_names, 'islet_label_sharpen'] =  spat.obs['dominant_sharpen']

src = spat.obsm['q05_cell_abundance_w_sf']          # DataFrame (rows = islet_after.obs_names)
filled = src.reindex(index=spatial.obs_names, fill_value=np.nan)  # or fill_value=np.nan

spatial.obsm['q05_cell_abundance_w_sf_islet'] = filled
spatial.obs.loc[spatial.obs['sample'] == 'U6-slice', 'group'] = 'Pre-T2D'
spatial.write('../data/YK_raw_spatial_islet_c2l_annot_hq.h5ad')